# HK_DEV_SHARED + HC7990 rerun control panel

A thin Colab control panel: mount Drive, clone the paper base, inspect contracts, then dispatch the versioned pipeline. It contains no duplicate cohort, Fi-NeMo, or DeepISA implementation. Every compute switch is off by default.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/Epromoter_grammar_decpher'
!test -d $REPO/.git || git clone https://github.com/JoneSu1/Epromoter_grammar_decpher.git $REPO
%cd $REPO
!git pull --ff-only
!git rev-parse --short HEAD


In [ ]:
# Restart the runtime if Colab asks after installation.
!pip -q install 'numpy<2' 'tensorflow==2.15.1' pandas h5py loguru bioframe finemo leidenalg igraph numba

import hashlib, json, os, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

CONFIG = f'{REPO}/configs/hk_dev_shared_rerun_config.json'
PIPELINE = f'{REPO}/scripts/hk_dev_shared_pipeline.py'
DATA_ROOT = '/content/drive/MyDrive/DeepEpromote/Drosophila'
OUT_ROOT = f'{DATA_ROOT}/HK_DEV_SHARED_rerun_202609'

def invoke(*args, capture=False):
    command = [sys.executable, PIPELINE, '--config', CONFIG, '--data-root', DATA_ROOT, '--output-root', OUT_ROOT, *args]
    print('\n$', ' '.join(command))
    return subprocess.run(command, check=True, text=True, capture_output=capture)

def show_states():
    state_dir = Path(OUT_ROOT) / 'state'
    rows = []
    for path in sorted(state_dir.glob('*.json')):
        payload = json.loads(path.read_text())
        rows.append({'stage': path.stem, 'status': payload.get('status'), 'completed_utc': payload.get('completed_utc'), 'n_outputs': len(payload.get('outputs', [])), 'fingerprint': str(payload.get('fingerprint', ''))[:12]})
    display(pd.DataFrame(rows) if rows else pd.DataFrame([{'stage': 'none yet'}]))

RUN_PLAN = {'prepare': False, 'admit_attributions': False, 'finemo': False, 'deepisa': False}
print('All compute switches are OFF:', RUN_PLAN)


## Gate 1 — sources, labels and fixed parameters

Expected: union 23,284; HC7990 7,990; sharing 17,380; HC-only 5,904; shared-only 15,294; both 2,086; non-distal 18,164; observed CAGE 10,082. Stop if any value differs.

In [ ]:
review = json.loads(invoke('inspect', capture=True).stdout)
display(Markdown('### Cohort contract'))
display(pd.DataFrame([review['observed_counts']]))
display(Markdown('### Five planned tracks'))
display(pd.DataFrame(review['tracks']).T[['scope', 'dictionary', 'expected_n', 'purpose']])
display(Markdown('### Fixed Fi-NeMo / attribution settings'))
display(pd.DataFrame([review['finemo'] | review['attribution_contract']]))
expected = {'union': 23284, 'hc7990': 7990, 'shared': 17380, 'hc7990_only': 5904, 'shared_only': 15294, 'hc7990_and_shared': 2086, 'union_non_distal': 18164, 'cage_observed_union': 10082}
assert {k: review['observed_counts'][k] for k in expected} == expected
print('Gate 1 PASS — source hashes and cohort contract agree.')


In [ ]:
# Set RUN_PLAN['prepare'] = True only after approving Gate 1.
if RUN_PLAN['prepare']:
    invoke('prepare')
else:
    print('prepare held for review')
show_states()


## Gate 2 — generated labelled manifests

Read-only QC: every selected sequence must be 249 bp and the DeepISA `region` key must be the canonical FASTA ID, never an ambiguous genomic coordinate.

In [ ]:
expected_tracks = {'s3_hk': 23284, 's3_dev': 23284, 'deepisa_hk': 18164, 'deepisa_dev': 18164, 'deepisa_cage': 10082}
rows = []
for track, expected_n in expected_tracks.items():
    frame = pd.read_csv(f'{OUT_ROOT}/manifests/{track}.tsv', sep='\t')
    rows.append({'track': track, 'n': len(frame), 'expected_n': expected_n, 'all_249bp': bool(frame.sequence.str.len().eq(249).all()), 'region_equals_canonical_id': bool((frame.region == frame.canonical_id).all()), **frame.combined_label.value_counts().to_dict()})
manifest_qc = pd.DataFrame(rows).fillna(0)
display(manifest_qc)
assert (manifest_qc.n == manifest_qc.expected_n).all() and manifest_qc.all_249bp.all() and manifest_qc.region_equals_canonical_id.all()
print('Gate 2 PASS — manifests are ready for both analysis branches.')


## Gate 3 — attribution provenance and Fi-NeMo admission

Generate five NPZ files by the reviewed model-specific dinucleotide DeepLIFT/SHAP procedure (100 backgrounds; batch 20). This notebook previews files and shapes; the base pipeline performs exact sequence-by-sequence admission.

In [ ]:
ATTRIBUTIONS = {track: f'{OUT_ROOT}/reviewed_attributions/{track}.npz' for track in expected_tracks}
attr_rows = []
for track, path in ATTRIBUTIONS.items():
    row = {'track': track, 'path': path, 'expected_n': expected_tracks[track], 'exists': os.path.exists(path)}
    if row['exists']:
        data = np.load(path, allow_pickle=False)
        row['keys'] = ','.join(data.files)
        row['sequences_shape'] = tuple(data['sequences'].shape) if 'sequences' in data.files else None
        row['hyp_scores_shape'] = tuple(data['hyp_scores'].shape) if 'hyp_scores' in data.files else None
    attr_rows.append(row)
attr_qc = pd.DataFrame(attr_rows)
display(attr_qc)
if RUN_PLAN['admit_attributions']:
    assert attr_qc.exists.all(), 'Missing attribution NPZ; do not admit a partial set.'
    for track, path in ATTRIBUTIONS.items():
        invoke('admit-attributions', '--track', track, '--source', path)
else:
    print('attribution admission held for review')
show_states()


## Gate 4 — Fi-NeMo scan plan

S3 HK/DEV scan the labelled union with separate standalone dictionaries. DeepISA HK/DEV/CAGE use the shared 24-bp atlas on their declared non-distal/observed manifests. Interrupted scan directories are archived before retry.

In [ ]:
cfg = json.loads(Path(CONFIG).read_text())
scan_plan = pd.DataFrame([{'track': track, 'n': expected_tracks[track], 'dictionary': cfg['sources'][review['tracks'][track]['dictionary']], 'lambda': cfg['finemo']['lambda'], 'max_steps': cfg['finemo']['max_steps']} for track in expected_tracks])
display(scan_plan)
if RUN_PLAN['finemo']:
    for track in expected_tracks:
        invoke('scan', '--track', track)
else:
    print('Fi-NeMo scans held for review')
show_states()


## Gate 5 — DeepISA automatic stage resume

Only HK, DEV and CAGE shared-atlas tracks enter this branch. `auto` resumes the first invalid state among preflight, single, combi, null and aggregate; each state stores output hashes.

In [ ]:
EP_ISA_SOURCE = f'{REPO}/paper_base/reproducibility_package/04_deepisa/scripts/Ep_ISA_NEW_src'
assert Path(EP_ISA_SOURCE, 'Ep_ISA_NEW').exists(), 'Paper base is incomplete; re-clone repository.'
display(pd.DataFrame([{'track': t, 'n': expected_tracks[t], 'model': cfg['sources']['deepcage_model' if t == 'deepisa_cage' else 'deepstarr_model'], 'resume_mode': 'auto'} for t in ('deepisa_hk', 'deepisa_dev', 'deepisa_cage')]))
if RUN_PLAN['deepisa']:
    for track in ('deepisa_hk', 'deepisa_dev', 'deepisa_cage'):
        invoke('deepisa', '--track', track, '--isa-source', EP_ISA_SOURCE, '--start-from', 'auto')
else:
    print('DeepISA held for review')
show_states()


## End-of-run handoff

All artifacts remain under `OUT_ROOT` on Drive: labelled manifests, reviewed attributions, Fi-NeMo scans, DeepISA tables, and `state/*.json`. Use canonical IDs and `combined_label` for local exploration rather than rebuilding cohorts.